# Gridsearch — LightGBM (ILI / ARI)

Runs the stage-1 (RMSE), freze best params of stage-1 and then gridsearch stage-2 (WIS) using Optuna.

## Setup

In [ ]:
# Optional: install dependencies (uncomment if running in a fresh env / Colab)
# %pip install "lightgbm>=4.3" "lightgbmlss>=0.2.5" "optuna>=3.6"



In [ ]:
import pandas as pd
from pathlib import Path
import sys
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import gridsearch_lgbm as gs


In [20]:
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "gridsearch_lgbm.py").exists():
            return p
    raise FileNotFoundError(
        "Could not locate repo root (src/gridsearch_lgbm.py not found above "
        f"{start}). Run this notebook from inside the MIGHTE-respicast-jointGBM checkout."
    )

ROOT_DIR = find_repo_root(Path.cwd())
print("Repo root:", ROOT_DIR)


Repo root: /home/nadillia/Documents/MIGHTE-respicast-jointGBM


In [ ]:
sys.path.insert(0, str(ROOT_DIR / "src"))

%load_ext autoreload
%autoreload 2

print("import OK")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
import OK


In [22]:
# Hub repo checkout (target-data + locations file). Also used below to refresh the canonical CSV.
HUB_DIR = ROOT_DIR / "RespiCast-SyndromicIndicators"
LOCATIONS_FILE = HUB_DIR / "supporting-files" / "locations_iso2_codes.csv"

if not LOCATIONS_FILE.exists():
    !git clone --depth 1 https://github.com/european-modelling-hubs/RespiCast-SyndromicIndicators.git "{HUB_DIR}"
else:
    !git -C "{HUB_DIR}" pull --ff-only


Already up to date.


## Config

Change `TARGET` to `"ILI"` or `"ARI"` and re-run from here down.

In [ ]:
TARGET = "ILI"  # or "ARI"

DATA_FILE = ROOT_DIR / "data" / "processed" / "respicast_long_latest.csv"
GT_FILE = ROOT_DIR / "google_preprocessing" / "data" / "processed" / "google_trends_preprocessed.csv"  # preprocessed (denoised+detrended)
ANCHOR = "2026-08-02"
EXCLUDE_COVID = True
INCLUDE_GT_LEAD = True  # adds a GT "nowcast" lag-1 feature (GT ~1 week ahead of the anchor); set False to disable

SEED = 4321
CUTOFF_Q = 0.75

# Keep these small (e.g. 5) while debugging, restore for a real search.
N_TRIALS_STAGE1 = 70
N_TRIALS_STAGE2 = 60

OUTPUT_PATH = ROOT_DIR / f"best_params_ablation_{TARGET.lower()}.json"

OWN_LAGS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 26, 52]
DONOR_LAGS = [1, 2, 3, 4, 8, 12]
DONOR_TOP_K = 4
OTHER_TOP_K = 2

OTHER = "ARI" if TARGET == "ILI" else "ILI"
TARGET_COL, OTHER_COL = gs.TARGET_COLUMN[TARGET], gs.TARGET_COLUMN[OTHER]
print(f"target={TARGET_COL!r}  secondary={OTHER_COL!r}")


In [24]:
# Rebuild the canonical long series from the hub checkout — same step forecast_backtest.py /
# forecast_prospective.py run before every forecast, so DATA_FILE reflects the latest target-data.

from build_long_timeseries import resolve_long_timeseries

canonical = resolve_long_timeseries(HUB_DIR)
DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
canonical.to_csv(DATA_FILE, index=False)
print(f"Refreshed {DATA_FILE}: {len(canonical)} rows, "
      f"max truth_date={pd.to_datetime(canonical['truth_date']).max().date()}")


Refreshed /home/nadillia/Documents/MIGHTE-respicast-jointGBM/data/processed/respicast_long_latest.csv: 22310 rows, max truth_date=2026-08-02


## Build feature matrix

In [ ]:
LOCATIONS = sorted(pd.read_csv(GT_FILE, usecols=["location"])["location"].astype(str).unique().tolist())
print(f"{len(LOCATIONS)} locations (from GT_FILE)")

if TARGET == "ILI":
    LOCATIONS = [loc for loc in LOCATIONS if loc not in {"CY", "IT"}]  # currently no ILI data for these countries 


df, y, feat_cols = gs.build_matrix(
    DATA_FILE, TARGET_COL, OTHER_COL, LOCATIONS, pd.to_datetime(ANCHOR),
    EXCLUDE_COVID, OWN_LAGS, DONOR_LAGS, DONOR_TOP_K, OTHER_TOP_K,
    gt_file=str(GT_FILE), include_gt_lead=INCLUDE_GT_LEAD,
)
print(f"{df.shape[0]} rows, {len(feat_cols)} features")
df.head()


## Stage 1 — RMSE search (num_leaves, learning_rate, min_child_samples, feature_fraction, rounds)

In [43]:
study_s1 = gs.run_stage1_study(
    gs.GT_VARIANT, df, y, feat_cols, SEED, N_TRIALS_STAGE1, CUTOFF_Q, verbose=True,
)
stage1_best = {**study_s1.best_params, "rounds": study_s1.best_trial.user_attrs["best_round"]}
print(f"best stage-1 RMSE = {study_s1.best_value:.3f}")
stage1_best


=== Stage 1 (RMSE) optimizing: gt_proc ===


Best trial: 47. Best value: 727.448: 100%|██████████| 70/70 [40:33<00:00, 34.76s/it]

best stage-1 RMSE = 727.448


{'num_leaves': 83,
 'learning_rate': 0.006541411439840936,
 'min_child_samples': 9,
 'feature_fraction': 0.6941494551885474,
 'rounds': 984}

In [44]:
# Inspect trial history if needed
study_s1.trials_dataframe().sort_values("value").head(10)


,number,value,datetime_start,datetime_complete,duration,params_feature_fraction,params_learning_rate,params_min_child_samples,params_num_leaves,user_attrs_best_round,state
47,47,727.447901,2026-08-10 21:12:38.180395,2026-08-10 21:13:19.876224,0 days 00:00:41.695829,0.694149,0.006541,9,83,984,COMPLETE
49,49,732.224711,2026-08-10 21:13:54.636546,2026-08-10 21:14:37.904115,0 days 00:00:43.267569,0.772105,0.005727,8,79,1000,COMPLETE
10,10,732.674914,2026-08-10 20:51:01.526961,2026-08-10 20:51:56.255168,0 days 00:00:54.728207,0.613948,0.005961,12,98,999,COMPLETE
61,61,732.942314,2026-08-10 21:22:07.159887,2026-08-10 21:22:45.074549,0 days 00:00:37.914662,0.897431,0.005804,8,80,1000,COMPLETE
44,44,734.687674,2026-08-10 21:10:28.137711,2026-08-10 21:11:07.173898,0 days 00:00:39.036187,0.616540,0.005905,9,83,999,COMPLETE
46,46,734.826113,2026-08-10 21:11:42.022736,2026-08-10 21:12:38.178904,0 days 00:00:56.156168,0.663653,0.006975,9,130,1000,COMPLETE
11,11,735.837250,2026-08-10 20:51:56.257396,2026-08-10 20:52:54.598814,0 days 00:00:58.341418,0.604796,0.005088,5,98,1000,COMPLETE
51,51,736.490255,2026-08-10 21:15:15.612891,2026-08-10 21:16:00.258129,0 days 00:00:44.645238,0.843471,0.005598,8,79,1000,COMPLETE
55,55,736.846952,2026-08-10 21:18:31.198426,2026-08-10 21:19:12.719688,0 days 00:00:41.521262,0.665508,0.006253,9,82,1000,COMPLETE
48,48,737.768536,2026-08-10 21:13:19.877720,2026-08-10 21:13:54.635230,0 days 00:00:34.757510,0.742968,0.006718,9,62,1000,COMPLETE


## Stage 2 — WIS search (stage2_rounds, lambda_l2, s2_min_child_samples), frozen on stage-1 rounds

In [ ]:
study_s2 = gs.run_stage2_study(
    gs.GT_VARIANT, df, y, feat_cols, stage1_best, SEED, N_TRIALS_STAGE2, CUTOFF_Q, verbose=True,
)
print(f"best stage-2 WIS = {study_s2.best_value:.4f}")
study_s2.best_params


=== Stage 2 (WIS) optimizing: gt_proc  (stage1 rounds frozen at 984) ===


  0%|          | 0/60 [00:00<?, ?it/s]/home/nadillia/Documents/MIGHTE-respicast-jointGBM/.venv/lib/python3.12/site-packages/lightgbmlss/utils.py:19: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  nan=float(torch.nanmean(predt)),


*** Using GaussianFrozenLocBounded (frozen mu, sigma in [0.1, 0.6]) ***


Best trial: 5. Best value: 127.491:  10%|█         | 6/60 [04:14<38:11, 42.43s/it]

In [ ]:
study_s2.trials_dataframe().sort_values("value").head(10)


## Pack + save

In [ ]:
best = {gs.GT_VARIANT: gs.pack(study_s1, study_s2)}
best


In [ ]:
import json

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, "w") as f:
    json.dump(best, f, indent=2)
print(f"Saved best params to {OUTPUT_PATH}")


## One-shot alternative

Equivalent to all the cells above, without the intermediate inspection —
useful once you're done debugging and just want the final params.

In [ ]:
# best = gs.run_gridsearch(
#     target=TARGET,
#     data_file=DATA_FILE,
#     locations_file=LOCATIONS_FILE,
#     anchor=ANCHOR,
#     gt_file=str(GT_FILE),
#     exclude_covid=EXCLUDE_COVID,
#     include_gt_lead=INCLUDE_GT_LEAD,
#     own_lags=OWN_LAGS,
#     donor_lags=DONOR_LAGS,
#     donor_top_k=DONOR_TOP_K,
#     other_top_k=OTHER_TOP_K,
#     cutoff_q=CUTOFF_Q,
#     seed=SEED,
#     n_trials_stage1=N_TRIALS_STAGE1,
#     n_trials_stage2=N_TRIALS_STAGE2,
#     output_path=OUTPUT_PATH,
# )
# best
